# 03 · Chia train/val/test, tách seed → bài đích, tạo tracks.parquet (ticket #6, #7)

**Input** (Add Input): Kaggle Dataset `mixnext-processed` v1 (`sessions_raw.parquet`, `tracks_raw.parquet`, `build_log.json` từ notebook 02).

**Output** (`/kaggle/working/processed/`): `tracks.parquet`, `sessions.parquet`, `eval_targets.parquet`, `split_report.json`, kèm bản sao 3 file của v1. Thư mục này chính là nội dung của `mixnext-processed` v2.

Bật Internet để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #6, #7 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "a"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
RAW_DIR = sorted(Path("/kaggle/input").rglob("sessions_raw.parquet"))[0].parent
OUT = Path("/kaggle/working/processed")
print("v1:", RAW_DIR, sorted(p.name for p in RAW_DIR.iterdir()))

In [ ]:
import time
from src.data.split import main

t0 = time.time()
result = main(["--raw-dir", str(RAW_DIR), "--out", str(OUT)])   # random_seed = 42; có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.iterdir()):
    print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")

## Đăng `mixnext-processed` v2

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Mở trang dataset `mixnext-processed` → **New Version**, thêm 7 file trong `processed/` của version vừa chạy. Nếu giao diện không cho chọn output của notebook thì tải 7 file về rồi upload.
3. Ghi chú version: "v2: tracks/sessions/eval_targets theo data-contract, random_seed 42".

**Kiểm tra chạy lại:** `split_report.json` có trường `fingerprint` (sha256 của ba bảng, tính theo đúng thứ tự dòng). Chạy lại notebook mà ra cùng `fingerprint` nghĩa là ra đúng cùng một tập.

## Kết quả (chạy trên Kaggle, CPU, 24/11/2026)

| | train | val | test |
|---|---:|---:|---:|
| số phiên | 73.924 | 9.241 | 9.241 |
| phiên trong eval_targets | | 9.212 | 9.206 |
| bị bỏ vì bài đích nằm trong seed | | 29 | 35 |
| bài đích không có trong train | | 0,0% | 0,0% |

`random_seed` = 42 · `fingerprint` = `c75e7c030dacf894807cbe8b5e4dd0f074b912c3e42e8e2fdea60bef5930ec3c`

Mọi bài đích đều có trong train, vì #5 chỉ giữ bài có ≥ 5 lượt trên toàn bộ dữ liệu.